In [31]:
import nltk
from nltk.corpus import sentiwordnet as swn
from nltk.corpus import wordnet as wn
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
import pandas as pd
import numpy as np
import argparse
import os

# Download required NLTK resources
nltk.download('sentiwordnet', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

True

In [32]:
# ─────────────────────────────────────────────
# 1. POS TAG CONVERSION
# ─────────────────────────────────────────────
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wn.ADJ
    elif treebank_tag.startswith('V'):
        return wn.VERB
    elif treebank_tag.startswith('N'):
        return wn.NOUN
    elif treebank_tag.startswith('R'):
        return wn.ADV
    else:
        return None

# ─────────────────────────────────────────────
# 2. SENTIWORDNET SCORE LOOKUP
# ─────────────────────────────────────────────
def get_sentiment_score(word, pos):
    lemmatizer = WordNetLemmatizer()
    lemma = lemmatizer.lemmatize(word, pos=pos)
    synsets = list(swn.senti_synsets(lemma, pos))
    if not synsets:
        return 0.0, 0.0, 1.0
    pos_score = np.mean([s.pos_score() for s in synsets])
    neg_score = np.mean([s.neg_score() for s in synsets])
    obj_score = np.mean([s.obj_score() for s in synsets])
    return pos_score, neg_score, obj_score

# ─────────────────────────────────────────────
# 3. NEGATION HANDLING
# ─────────────────────────────────────────────
NEGATION_WORDS = {
    "not", "no", "never", "neither", "nor", "nobody",
    "nothing", "nowhere", "hardly", "scarcely", "barely",
    "n't", "nt", "without", "cannot", "can't", "won't",
    "isn't", "aren't", "wasn't", "weren't", "doesn't",
    "don't", "didn't", "hasn't", "haven't", "hadn't"
}

def apply_negation(tokens):
    WINDOW = 3
    negated_flags = [False] * len(tokens)
    neg_countdown = 0
    for i, token in enumerate(tokens):
        if token.lower() in NEGATION_WORDS:
            neg_countdown = WINDOW
        elif neg_countdown > 0:
            negated_flags[i] = True
            neg_countdown -= 1
    return list(zip(tokens, negated_flags))

# ─────────────────────────────────────────────
# 4. MAIN CLASSIFIER
# ─────────────────────────────────────────────
def sentiwordnet_classify(text, threshold=0.05):
    tokens = word_tokenize(text.lower())
    tagged = pos_tag(tokens)
    token_neg_pairs = apply_negation(tokens)

    total_pos, total_neg = 0.0, 0.0
    scored_words = 0

    for (word, treebank_pos), (_, is_negated) in zip(tagged, token_neg_pairs):
        wn_pos = get_wordnet_pos(treebank_pos)
        if wn_pos is None:
            continue
        p, n, _ = get_sentiment_score(word, wn_pos)
        if p == 0 and n == 0:
            continue
        if is_negated:
            p, n = n, p
        total_pos += p
        total_neg += n
        scored_words += 1

    if scored_words == 0:
        return {"label": "Neutral", "net_score": 0.0,
                "pos_score": 0.0, "neg_score": 0.0}

    avg_pos = total_pos / scored_words
    avg_neg = total_neg / scored_words
    net     = avg_pos - avg_neg

    if net > threshold:
        label = "Positive"
    elif net < -threshold:
        label = "Negative"
    else:
        label = "Neutral"

    return {
        "label":     label,
        "net_score": round(net, 4),
        "pos_score": round(avg_pos, 4),
        "neg_score": round(avg_neg, 4),
    }

# ─────────────────────────────────────────────
# 5. BATCH CLASSIFICATION
# ─────────────────────────────────────────────
def classify_dataframe(df, text_column="text", threshold=0.05):
    results = df[text_column].apply(
        lambda t: sentiwordnet_classify(str(t), threshold)
    )
    df = df.copy()
    df["swn_label"]     = results.apply(lambda r: r["label"])
    df["swn_net_score"] = results.apply(lambda r: r["net_score"])
    df["swn_pos_score"] = results.apply(lambda r: r["pos_score"])
    df["swn_neg_score"] = results.apply(lambda r: r["neg_score"])
    return df

# ─────────────────────────────────────────────
# 6. EVALUATION HELPER
# ─────────────────────────────────────────────
def evaluate(df, true_col, pred_col="swn_label"):
    from sklearn.metrics import classification_report, confusion_matrix
    import io
    import sys

    # Normalize case for both columns
    true = df[true_col].str.lower().str.strip()
    pred = df[pred_col].str.lower().str.strip()

    # Capture the classification report
    report = classification_report(true, pred)
    
    # Get confusion matrix
    labels = sorted(true.unique())
    cm = confusion_matrix(true, pred, labels=labels)
    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    
    return report, cm_df


In [33]:
# Load the dataset
df = pd.read_csv('../data/STEAM_GAMES_CLEAN_GTLabel.csv')

# Display basic info
print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")


# Classify the reviews using SentiWordNet
df_classified = classify_dataframe(df, text_column='review_text', threshold=0.0)

print("\nClassification completed!")

# Evaluate the classifier against ground truth
report, cm_df = evaluate(df_classified, true_col='final_label', pred_col='swn_label')

print("=== SentiWordNet Classifier — Evaluation ===\n")
print(report)
print("Confusion Matrix:")
print(cm_df)

# Save the classified data
output_path = '../data/SWresults.csv'
df_classified.to_csv(output_path, index=False)
print(f"\nResults saved to: {output_path}")

Dataset loaded successfully!
Shape: (200, 22)
Columns: ['Unnamed: 0.2', 'Unnamed: 0.1', 'Unnamed: 0', 'game_name', 'app_id', 'review_text', 'review_length', 'hours_played', 'review_date', 'owners', 'developers', 'publishers', 'genres', 'platforms', 'categories', 'release_date', 'price', 'label_1', 'label_2', 'label_3', 'final_label', 'category']

Classification completed!
=== SentiWordNet Classifier — Evaluation ===

              precision    recall  f1-score   support

    negative       0.47      0.53      0.50        51
     neutral       0.86      0.65      0.74        48
    positive       0.71      0.74      0.72       101

    accuracy                           0.67       200
   macro avg       0.68      0.64      0.65       200
weighted avg       0.68      0.67      0.67       200

Confusion Matrix:
          negative  neutral  positive
negative        27        2        22
neutral          8       31         9
positive        23        3        75

Results saved to: ../data/S